In [ ]:
#meta for myFiftyoneComputerVision 6/2/2025 AML MLP 0 Load Data - Get PDFs from Azure Storage File
#started from my-azure-2025/sample_Connect_to_FileStorage.ipynb

#env Azure cloud
# default Python 3.8 - AzureML
#EXPDCopilot prompt "how to access files in Azure File storage with python"
#pip install azure-storage-file-share
#pip install azure-storage-file


#Reference (real)
#Configure a connection string for an Azure storage account (subtitle)
# refer to https://learn.microsoft.com/en-us/azure/storage/common/storage-configure-connection-string#configure-a-connection-string-for-an-azure-storage-account
# Original title Configure Azure Storage connection strings
# refer to https://learn.microsoft.com/en-us/azure/storage/common/storage-configure-connection-string

#Listing Contents of a directory
# refer to https://learn.microsoft.com/en-us/python/api/overview/azure/storage-file-share-readme?view=azure-python#listing-contents-of-a-directory
# Original title Azure Storage File Share client library for Python - version 12.21.0
# refer to https://learn.microsoft.com/en-us/python/api/overview/azure/storage-file-share-readme?view=azure-python

#history
#6/2/2025 GET INVOICES PDFs FROM AZURE STORAGE FILE
#      Successfully loaded from with 'sample-invoices-999' $config
#      Sample dataset: Mendeley Data, Samples of electronic invoices
#        refer to https://data.mendeley.com/datasets/tnj49gpmtz/2 

#6/9/2025 MISC CLEANUP (SAMPLE INVOICES 250) 
#      Rename files and folders
#      was DOC-SAMPLE-INVOICES-999_PDFs, DOC-SAMPLE-INVOICES-250_Images, df_sample_invoices_250_metadata.parquet
#      now PDFs_SAMPLE-INVOICES-250, IMAGES_SAMPLE-INVOICES-250, df_sample-invoices-250_metadata.parquet
# did not run


#$private

In [ ]:
import os

from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
from azure.storage.fileshare import ShareServiceClient, ShareClient, ShareDirectoryClient, ShareFileClient
from azure.storage.file import FileService
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient


# Source Documents Dataset
with a goal to start with original PDFs and convert PDFs to Images
## 0. Load Original Data
Get PDFs from Azure File Storage

### 0.1 Connect to File Storage
Azure Storage File

In [ ]:
#Retrieve a Secret $config $private
KEY_VAULT_NAME = "[KV_NAME]"
SECRET_NAME = "conn-filestorage-[STORAGE_ACCOUNT_NAME]"

kv_uri = f"https://{KEY_VAULT_NAME}.vault.azure.net"

# Use DefaultAzureCredential to authenticate
credential = DefaultAzureCredential()
client = SecretClient(vault_url=kv_uri, credential=credential)

retrieved_secret = client.get_secret(SECRET_NAME)

In [ ]:
#from azure.storage.fileshare import ShareServiceClient
#$config
# Replace 'your_connection_string' with the actual connection string from your Azure Storage account
#connection_string = "DefaultEndpointsProtocol=https;AccountName=your_account_name;AccountKey=your_account_key;EndpointSuffix=core.windows.net"
CONN = retrieved_secret.value

# Create the ShareServiceClient
service_client = ShareServiceClient.from_connection_string(CONN)

# Example: List all shares in the storage account
shares = service_client.list_shares()
for share in shares:
    print(share.name) #out: FILESHARE_NAME ...


### 0.2 Download PDFs
from Azure Storage FileShare to a local directory

In [ ]:
#EXPDCopilot "how to iteratively download files from Azure Storage FileShare directory"
# Initialize a connection to the Azure Storage File account
file_service = FileService(connection_string=CONN)

# Specify the share name and directory path #$config
FILESHARE_NAME = 'anyac-fileshare-try'
FILESHARE_DIR_SRC = 'sample-invoices-999' 
LOCAL_DIR_TGT = 'PDFs_SAMPLE-INVOICES-250'
#N = 250 # number of documents to process

# List all files in the directory
files_list = file_service.list_directories_and_files(FILESHARE_NAME, FILESHARE_DIR_SRC) #azure.storage.common.models.ListGenerator
l_files = list(files_list) #class list

# Create a directory to save the downloaded files
##os.makedirs("downloaded_files", exist_ok=True) => LOCAL_DIR_TGT #$config

# Download all files from the directory
for item in l_files: #[:N]:
    file_path = os.path.join(LOCAL_DIR_TGT, item.name)
    
    # Download the file
    file_service.get_file_to_path(FILESHARE_NAME, FILESHARE_DIR_SRC, item.name, file_path)

print("All files have been downloaded successfully.")


In [ ]:
mystop

## Xtra

In [ ]:
#$xtra Listing Contents of a directory
# refer to https://learn.microsoft.com/en-us/python/api/overview/azure/storage-file-share-readme?view=azure-python#listing-contents-of-a-directory
from azure.storage.fileshare import ShareDirectoryClient

parent_dir = ShareDirectoryClient.from_connection_string(conn_str=connection_string, share_name=SHARE_NAME, directory_path=FILESHARE_DIR_SRC)

my_list = list(parent_dir.list_directories_and_files())
my_list[0]

In [ ]:
#$xtra ?another way to download all files (or all PDFs from a container)
def upload_file(connection_string, share_name, dir_name, file_name, file_content):
    file_client = ShareFileClient.from_connection_string(connection_string, share_name, dir_name, file_name)
    file_client.upload_file(file_content)

def download_file(connection_string, share_name, dir_name, file_name):
    file_client = ShareFileClient.from_connection_string(connection_string, share_name, dir_name, file_name)
    return file_client.download_file().readall()

def delete_file(connection_string, share_name, dir_name, file_name):
    file_client = ShareFileClient.from_connection_string(connection_string, share_name, dir_name, file_name)
    file_client.delete_file()
